# Round 0 Data Exploration
Basic exploration of prices and trades data across day -2 and day -1.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Load and combine prices
prices = pd.concat([
    pd.read_csv('prices_round_0_day_-2.csv', sep=';'),
    pd.read_csv('prices_round_0_day_-1.csv', sep=';'),
], ignore_index=True)

# Load and combine trades
trades = pd.concat([
    pd.read_csv('trades_round_0_day_-2.csv', sep=';'),
    pd.read_csv('trades_round_0_day_-1.csv', sep=';'),
], ignore_index=True)

print('Prices shape:', prices.shape)
print('Trades shape:', trades.shape)

## Prices — Overview

In [ ]:
prices.head(10)

In [ ]:
print('Products:', prices['product'].unique())
print('Days:', sorted(prices['day'].unique()))
print('Timestamp range:', prices['timestamp'].min(), '→', prices['timestamp'].max())
print()
prices.describe()

### Mid-price stats by product and day

In [ ]:
prices.groupby(['product', 'day'])['mid_price'].agg(['mean', 'std', 'min', 'max']).round(2)

### Bid-ask spread (level 1)

In [ ]:
prices['spread_1'] = prices['ask_price_1'] - prices['bid_price_1']
prices.groupby(['product', 'day'])['spread_1'].agg(['mean', 'std', 'min', 'max']).round(2)

### Mid-price over time

In [ ]:
products = prices['product'].unique()
days = sorted(prices['day'].unique())

fig, axes = plt.subplots(len(products), len(days), figsize=(14, 4 * len(products)), sharey='row')
if len(products) == 1:
    axes = [axes]

for r, product in enumerate(products):
    for c, day in enumerate(days):
        ax = axes[r][c]
        subset = prices[(prices['product'] == product) & (prices['day'] == day)]
        ax.plot(subset['timestamp'], subset['mid_price'], lw=1)
        ax.fill_between(subset['timestamp'], subset['bid_price_1'], subset['ask_price_1'], alpha=0.2, label='Bid-Ask spread')
        ax.set_title(f'{product} | Day {day}')
        ax.set_xlabel('Timestamp')
        ax.set_ylabel('Price')

plt.suptitle('Mid-price with Level-1 Bid-Ask Band', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Trades — Overview

In [ ]:
trades.head(10)

In [ ]:
print('Symbols:', trades['symbol'].unique())
print('Currency:', trades['currency'].unique())
print('Timestamp range:', trades['timestamp'].min(), '→', trades['timestamp'].max())
print()
trades.describe()

### Trade stats by symbol

In [ ]:
trades.groupby('symbol').agg(
    num_trades=('price', 'count'),
    total_volume=('quantity', 'sum'),
    avg_price=('price', 'mean'),
    std_price=('price', 'std'),
    min_price=('price', 'min'),
    max_price=('price', 'max'),
    avg_qty=('quantity', 'mean'),
).round(2)

### Trade price distribution

In [ ]:
symbols = trades['symbol'].unique()
fig, axes = plt.subplots(1, len(symbols), figsize=(7 * len(symbols), 4))
if len(symbols) == 1:
    axes = [axes]

for ax, sym in zip(axes, symbols):
    sub = trades[trades['symbol'] == sym]
    ax.hist(sub['price'], bins=30, edgecolor='black', alpha=0.75)
    ax.set_title(f'{sym} — Trade Price Distribution')
    ax.set_xlabel('Price')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

### Trade volume over time

In [ ]:
fig, axes = plt.subplots(len(symbols), 1, figsize=(14, 4 * len(symbols)))
if len(symbols) == 1:
    axes = [axes]

for ax, sym in zip(axes, symbols):
    sub = trades[trades['symbol'] == sym].sort_values('timestamp')
    ax.scatter(sub['timestamp'], sub['price'], s=sub['quantity'] * 4, alpha=0.5, label='Trade (size ~ qty)')
    ax.set_title(f'{sym} — Trade Prices (bubble size = quantity)')
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('Price')
    ax.legend()

plt.tight_layout()
plt.show()